# 6×6 Protocol-Specific Fine-Tuning

For each driving protocol H(t) ∈ {GRF, tanh, Gaussian}, fine-tune the pre-trained
propagator U_θ(t; H) using sparse DMRG observable measurements from a **seen**
initial state (seed=42).  Then test whether the updated propagator improves
predictions for an **unseen** initial state (seed=69) under the same protocol.

Fine-tuning data : `dmrg_6x6_chi128.pkl`  (seed=42, 3 protocols)
Generalization test : `dmrg_6x6_chi128_FineTuning.pkl`  (seed=69, same protocols)

### Surrogate loss
Sparse measurements at K=4 time points.  Observable-matching via score-function estimator:
$$\mathcal{L}^\text{surr} = \underbrace{\sum_k r_k^x \overline{X^\text{loc}_k}}_{\text{direct}} + \underbrace{\sum_k \overline{r_k^x (X^\text{loc}_k - \bar X_k)\log p} + r_k^{mz}(\cdots) + r_k^{zz}(\cdots)}_{\text{score-function}}$$
where residuals $r_k = \langle O\rangle^\theta_{t_k} - O^\star_k$ are stop-gradients.


In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import sys, pickle, time, functools
from pathlib import Path

import jax
import jax.numpy as jnp
from jax import lax, random
import equinox as eqx
import numpy as np
import optax
import matplotlib.pyplot as plt
from tqdm import tqdm
import __main__

sys.path.insert(0, str(Path(".").resolve()))
import learn_propagator as lp

print("JAX devices:", jax.devices())

In [ ]:
# ── System constants ───────────────────────────────────────────────────────────
LX, LY     = 6, 6
N_SPINS    = LX * LY       # 36
SITE_ORDER = "snake"
T_STEPS    = 200
T_MAX      = 0.5
DT         = T_MAX / float(T_STEPS - 1)
PROTOCOLS  = ["GRF", "tanh", "Gaussian"]

# ── Load model ─────────────────────────────────────────────────────────────────
CHECKPOINT = Path("learn_propagator_latest_checkpoint_6x6.pkl")

def _set_checkpoint_aliases():
    for name in ["PropagatorNOQS", "FNO1D", "SpectralConv1D",
                 "SpinDecoderBlock", "Embedding", "LayerNorm",
                 "Linear", "MultiHeadDotProductAttention"]:
        if hasattr(lp, name):
            setattr(__main__, name, getattr(lp, name))

_set_checkpoint_aliases()
payload     = lp.load_resume_payload(CHECKPOINT)
saved_model = payload["best_model"]
saved_cfg   = payload.get("config", {})
n_spins     = int(saved_model.n_spins)

dt               = T_MAX / float(T_STEPS - 1)
fixed_M0_mode    = getattr(saved_model, "fixed_M0_mode",    saved_cfg.get("fixed_M0_mode",    "random_unit_rows"))
fixed_M0_seed    = getattr(saved_model, "fixed_M0_seed",    42)
phase_mode       = getattr(saved_model, "phase_mode",       saved_cfg.get("phase_mode",       "raw"))
self_kv_cache    = bool(getattr(saved_model, "self_kv_cache", saved_cfg.get("self_kv_cache",   False)))
integration_rule = getattr(saved_model, "integration_rule", saved_cfg.get("integration_rule", "simpson"))
fixed_seed       = fixed_M0_seed if fixed_M0_mode not in (None, "canonical_basis", "learnable_diff") else None

model_template = lp.PropagatorNOQS(
    int(saved_model.embed_dim),
    int(saved_model.num_heads),
    int(saved_model.num_layers),
    int(getattr(saved_model, "mlp_mult", saved_cfg.get("mlp_mult", 4))),
    int(saved_model.fno_modes),
    int(saved_model.fno_width),
    int(saved_model.ctx_tokens),
    int(saved_model.in_fields),
    phase_mode=phase_mode,
    self_kv_cache=self_kv_cache,
    fixed_M0_mode=fixed_M0_mode,
    fixed_M0_seed=fixed_seed,
    n_spins=n_spins,
    integration_rule=integration_rule,
    dt=dt,
    key=random.PRNGKey(int(fixed_M0_seed) if fixed_M0_seed is not None else 0),
)
base_model = lp.combine_model(lp.upgrade_resume_model(saved_model, model_template), model_template)

bonds_i_np, bonds_j_np = lp.make_lattice_bonds(LX, LY, SITE_ORDER)
bonds_i_jnp = jnp.array(bonds_i_np, dtype=jnp.int32)
bonds_j_jnp = jnp.array(bonds_j_np, dtype=jnp.int32)
N_BONDS     = len(bonds_i_np)    # 60 for 6×6 OBC
time_grid   = np.linspace(0.0, T_MAX, T_STEPS, dtype=np.float32)

_snake_coords     = lp.lattice_visit_order(LX, LY, "snake")
snake_to_rowmajor = np.array([y * LX + x for (x, y) in _snake_coords], dtype=np.int64)

print(f"n_spins={n_spins}  dt={dt:.5f}  T_MAX={T_MAX}  T_STEPS={T_STEPS}  N_BONDS={N_BONDS}")
print(f"model: d={base_model.embed_dim} heads={base_model.num_heads} "
      f"layers={base_model.num_layers} fno_width={base_model.fno_width} ctx={base_model.ctx_tokens}")

# ── Load DMRG data ──────────────────────────────────────────────────────────────
def load_dmrg(path):
    """Load DMRG pkl; reorder bits from row-major to snake order."""
    with open(path, "rb") as f:
        raw = pickle.load(f)
    data = {}
    for proto in PROTOCOLS:
        d = raw[proto]
        bits_rowmajor = np.array(d["bits"], dtype=np.int32)
        bits_snake    = bits_rowmajor[snake_to_rowmajor].astype(np.int32)
        hx_d = np.array(d["hx"], dtype=np.float32)
        hz_d = np.array(d["hz"], dtype=np.float32)
        zz_d = np.array(d["zz"], dtype=np.float32)
        mx_d = np.array(d["mx"], dtype=np.float32)
        mz_d = np.array(d["mz"], dtype=np.float32)
        data[proto] = {
            "bits_rowmajor": bits_rowmajor,
            "bits_snake":    bits_snake,
            "bitstring":     d["bitstring"],
            "t":   np.array(d["t"],   dtype=np.float32),
            "hx":  hx_d,  "hz":  hz_d,
            "h":   np.array(d["h"],   dtype=np.float32),
            "mx":  mx_d,  "mz":  mz_d,  "zz":  zz_d,
            "ent": np.array(d["ent"], dtype=np.float32),
        }
    return data

dmrg_train = load_dmrg("dmrg_6x6_chi128.pkl")             # seed=42 — FT targets
dmrg_test  = load_dmrg("dmrg_6x6_chi128_FineTuning.pkl")  # seed=69 — UNSEEN initial states

print("\ndmrg_train (seed=42):")
for p in PROTOCOLS:
    print(f"  {p}: |{dmrg_train[p]['bitstring']}⟩")
print("\ndmrg_test (seed=69, UNSEEN):")
for p in PROTOCOLS:
    print(f"  {p}: |{dmrg_test[p]['bitstring']}⟩")

In [ ]:
# ── Conditional sampler (same interface as 6by6_benchmark.ipynb) ──────────────

@eqx.filter_jit
def sample_given_beta(key, model_ref, ctx, beta_bits_snake, n_samples):
    """Sample n_samples alpha configs from |U(alpha|beta,t)|^2.
    ctx : (ctx_tokens, embed_dim)  — context for a single time step.
    """
    n_sp      = beta_bits_snake.shape[0]
    ctx_batch = jnp.broadcast_to(ctx[None], (n_samples, ctx.shape[0], ctx.shape[1]))
    sigmas    = jnp.broadcast_to(beta_bits_snake[None, :].astype(jnp.int32), (n_samples, n_sp))
    token_beta = jnp.array([0, 1, 0, 1], dtype=jnp.int32)

    def body(carry, i):
        s, rng = carry
        logits = model_ref.logits_from_tokens(s, ctx_batch)[:, i, :]
        beta_i = beta_bits_snake[i]
        valid  = (token_beta == beta_i)
        masked = jnp.where(valid[None, :], logits, jnp.full_like(logits, -1e9))
        rng, sub = random.split(rng)
        s_i = random.categorical(sub, masked).astype(jnp.int32)
        return (s.at[:, i].set(s_i), rng), None

    (sigmas, _), _ = lax.scan(body, (sigmas, key), jnp.arange(n_sp, dtype=jnp.int32))
    return sigmas


@eqx.filter_jit
def estimate_obs_mc(sigmas, ctx, model_ref, bi, bj, h_x_t, h_z_t):
    """Estimate mx, mz, zz, energy from MC-sampled alpha states."""
    n_s, n_sp = sigmas.shape
    ctx_batch = jnp.broadcast_to(ctx[None], (n_s, ctx.shape[0], ctx.shape[1]))
    alpha_bits = sigmas // 2
    alpha_pm   = 1.0 - 2.0 * alpha_bits.astype(jnp.float32)
    mz = jnp.mean(alpha_pm)
    zz = jnp.mean(alpha_pm[:, bi] * alpha_pm[:, bj])
    log_u = model_ref.log_psi_from_tokens(sigmas, ctx_batch)

    def ratio_fn(i):
        s_flip  = lp.sigma_flip_alpha(sigmas, i)
        lp_flip = model_ref.log_psi_from_tokens(s_flip, ctx_batch)
        return jnp.real(jnp.exp(lp_flip - log_u))

    all_ratios = jax.vmap(ratio_fn)(jnp.arange(n_sp, dtype=jnp.int32))
    mx = jnp.mean(jnp.sum(all_ratios, axis=0)) / float(n_sp)
    energy = -lp.J_zz * (float(bi.shape[0]) / float(n_sp)) * zz - h_z_t * mz - h_x_t * mx
    return mx, mz, zz, energy


def run_mc_benchmark(model_ref, proto_data, n_samples=2048, seed=42, verbose=True):
    """Full-trajectory MC evaluation for one protocol and initial state."""
    d    = proto_data
    h    = jnp.asarray(d["h"],          dtype=jnp.float32)
    beta = jnp.asarray(d["bits_snake"], dtype=jnp.int32)

    m_traj, _ = model_ref.encode_field(h[None, :, :])
    m_traj    = m_traj[0]   # (T_STEPS, ctx, D)

    mx_list, mz_list, zz_list, energy_list = [], [], [], []
    rng = random.PRNGKey(seed)

    for t_idx in range(T_STEPS):
        rng, subkey = random.split(rng)
        ctx_t  = m_traj[t_idx]
        h_x_t  = jnp.float32(d["hx"][t_idx])
        h_z_t  = jnp.float32(d["hz"][t_idx])
        sigmas = sample_given_beta(subkey, model_ref, ctx_t, beta, n_samples)
        mx, mz, zz, energy = estimate_obs_mc(
            sigmas, ctx_t, model_ref, bonds_i_jnp, bonds_j_jnp, h_x_t, h_z_t
        )
        mx_list.append(float(mx));  mz_list.append(float(mz))
        zz_list.append(float(zz));  energy_list.append(float(energy))
        if verbose and (t_idx % 50 == 0 or t_idx == T_STEPS - 1):
            print(f"  t={time_grid[t_idx]:.3f} ({t_idx+1}/{T_STEPS})"
                  f"  mx={mx:.4f}  mz={mz:.4f}  zz={zz:.4f}", end="\r")
    if verbose:
        print()
    return {
        "mx_pred":     np.array(mx_list,     dtype=np.float32),
        "mz_pred":     np.array(mz_list,     dtype=np.float32),
        "zz_pred":     np.array(zz_list,     dtype=np.float32),
        "energy_pred": np.array(energy_list, dtype=np.float32),
    }

def run_mc_benchmark_M(M_traj, model_ref, proto_data, n_samples=2048, seed=42, verbose=True):
    """Full-trajectory MC evaluation using a pre-computed context trajectory M_traj.
    model_ref decoder weights are frozen; M_traj encodes the propagator U(t).
    """
    d    = proto_data
    beta = jnp.asarray(d["bits_snake"], dtype=jnp.int32)

    mx_list, mz_list, zz_list, energy_list = [], [], [], []
    rng = random.PRNGKey(seed)

    for t_idx in range(T_STEPS):
        rng, subkey = random.split(rng)
        ctx_t  = jnp.asarray(M_traj[t_idx])
        h_x_t  = jnp.float32(d["hx"][t_idx])
        h_z_t  = jnp.float32(d["hz"][t_idx])
        sigmas = sample_given_beta(subkey, model_ref, ctx_t, beta, n_samples)
        mx, mz, zz, energy = estimate_obs_mc(
            sigmas, ctx_t, model_ref, bonds_i_jnp, bonds_j_jnp, h_x_t, h_z_t
        )
        mx_list.append(float(mx));  mz_list.append(float(mz))
        zz_list.append(float(zz));  energy_list.append(float(energy))
        if verbose and (t_idx % 50 == 0 or t_idx == T_STEPS - 1):
            print(f"  t={time_grid[t_idx]:.3f} ({t_idx+1}/{T_STEPS})"
                  f"  mx={mx:.4f}  mz={mz:.4f}  zz={zz:.4f}", end="\r")
    if verbose:
        print()
    return {
        "mx_pred":     np.array(mx_list,     dtype=np.float32),
        "mz_pred":     np.array(mz_list,     dtype=np.float32),
        "zz_pred":     np.array(zz_list,     dtype=np.float32),
        "energy_pred": np.array(energy_list, dtype=np.float32),
    }


## Pre-FT Baseline

Load pre-computed predictions for the **seen** initial state (seed=42) from `6by6_results.npz`.
Run MC benchmark on the **unseen** initial state (seed=69) using the base model.

In [ ]:
N_SAMPLES_EVAL = 2048

# Seen state (seed=42): load from the benchmark NPZ
npz = np.load("6by6_results.npz")
pre_train = {}
for proto in PROTOCOLS:
    pre_train[proto] = {
        "mx_pred":     npz[f"{proto}_mx_pred"],
        "mz_pred":     npz[f"{proto}_mz_pred"],
        "zz_pred":     npz[f"{proto}_zz_pred"],
        "energy_pred": npz[f"{proto}_energy_pred"],
    }
print("Loaded pre-FT seen-state results from 6by6_results.npz")

# Unseen state (seed=69): run or load from cache
CACHE_PRE_TEST = Path("ft_pre_test_results.pkl")
if CACHE_PRE_TEST.exists():
    with open(CACHE_PRE_TEST, "rb") as f:
        pre_test = pickle.load(f)
    print(f"Loaded pre-FT unseen-state results from {CACHE_PRE_TEST}")
else:
    pre_test = {}
    for proto in PROTOCOLS:
        print(f"[Pre-FT / unseen / {proto}] evaluating...")
        t0 = time.time()
        pre_test[proto] = run_mc_benchmark(
            base_model, dmrg_test[proto], n_samples=N_SAMPLES_EVAL, seed=0
        )
        print(f"  {proto} done in {time.time()-t0:.1f}s")
    with open(CACHE_PRE_TEST, "wb") as f:
        pickle.dump(pre_test, f)
    print(f"Saved → {CACHE_PRE_TEST}")

# Metrics summary
print(f"\n{'Protocol':<10} {'State':<10}  {'MAE mx':>8}  {'MAE mz':>8}  {'MAE zz':>8}")
print("-" * 52)
for proto in PROTOCOLS:
    for tag, preds, refs in [
        ("seen",   pre_train[proto], dmrg_train[proto]),
        ("unseen", pre_test[proto],  dmrg_test[proto]),
    ]:
        mae_mx = float(np.mean(np.abs(preds["mx_pred"] - refs["mx"])))
        mae_mz = float(np.mean(np.abs(preds["mz_pred"] - refs["mz"])))
        mae_zz = float(np.mean(np.abs(preds["zz_pred"] - refs["zz"])))
        print(f"{proto:<10} {tag:<10}  {mae_mx:>8.5f}  {mae_mz:>8.5f}  {mae_zz:>8.5f}")

In [ ]:
# ── Fine-tuning hyperparameters ─────────────────────────────────────────────
FT_STEPS   = 200
FT_LR      = 1e-3
FT_MC      = 512     # MC samples per time step in each batch
T_BATCH    = 20      # random time steps sampled per gradient step
REG_LAMBDA = 0.05    # L2 pull toward base model weights

print(f"FT_STEPS={FT_STEPS}  FT_LR={FT_LR}  FT_MC={FT_MC}  "
      f"T_BATCH={T_BATCH}  REG_LAMBDA={REG_LAMBDA}")
print(f"Expected time-step visits per training run: "
      f"{FT_STEPS * T_BATCH / T_STEPS:.0f}x per time step")

def run_protocol_finetune_decoder(base_model, proto, dmrg_data,
                                   n_steps=FT_STEPS, lr=FT_LR, n_mc=FT_MC,
                                   t_batch=T_BATCH, reg_lambda=REG_LAMBDA):
    """Fine-tune the transformer (decoder) weights with M_traj frozen.

    M_traj is computed once from the FNO encoder and never updated.
    All decoder weights (pos_embed, token_embed, blocks, heads) are learnable.
    Same mini-batch schedule and mx+zz surrogate loss as the M_traj variant.
    L2 regularization pulls decoder weights toward the pre-FT base model.
    """
    d        = dmrg_data[proto]
    h_jnp    = jnp.asarray(d["h"],          dtype=jnp.float32)
    beta_jnp = jnp.asarray(d["bits_snake"], dtype=jnp.int32)
    tgt_mx_all = jnp.asarray(d["mx"], dtype=jnp.float32)
    tgt_zz_all = jnp.asarray(d["zz"], dtype=jnp.float32)

    # Pre-compute M_traj once and freeze it
    M_traj = jax.lax.stop_gradient(
        base_model.encode_field(h_jnp[None])[0][0]   # (T_STEPS, ctx, D)
    )
    print(f"  M_traj frozen: {M_traj.shape}  "
          f"(optimising decoder over all {T_STEPS} time steps, batch={t_batch})")

    model     = base_model
    optimizer = optax.chain(optax.clip_by_global_norm(0.5), optax.adam(lr))
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    mse_hist   = []
    best_mse   = float("inf")
    best_model = model

    # Frozen reference for L2 regularization — leaves list is static
    base_leaves = jax.tree_util.tree_leaves(eqx.filter(base_model, eqx.is_array))

    @eqx.filter_jit
    def ft_step(model, opt_state, t_batch_idx, all_sigmas, tgt_mx_b, tgt_zz_b):
        def loss_fn(model):
            M_batch = M_traj[t_batch_idx]   # (B, ctx, D) — frozen

            def loss_one_t(carry, inp):
                M_t, mx_k, zz_k, sigmas_k = inp
                M_flat = jnp.broadcast_to(M_t[None], (n_mc,) + M_t.shape)
                sigmas = lax.stop_gradient(sigmas_k)

                log_u    = model.log_psi_from_tokens(sigmas, M_flat)
                log_prob = 2.0 * jnp.real(log_u)

                alpha_pm = 1.0 - 2.0 * (sigmas // 2).astype(jnp.float32)
                zz_loc   = jnp.mean(
                    alpha_pm[:, bonds_i_jnp] * alpha_pm[:, bonds_j_jnp], axis=-1
                )

                def x_ratio_fn(site):
                    s_flip     = lp.sigma_flip_alpha(sigmas, site)
                    log_u_flip = model.log_psi_from_tokens(s_flip, M_flat)
                    return jnp.real(jnp.exp(log_u_flip - log_u))

                all_ratios = lax.map(
                    jax.checkpoint(x_ratio_fn), jnp.arange(N_SPINS, dtype=jnp.int32)
                )
                mx_loc  = jnp.sum(all_ratios, axis=0) / float(N_SPINS)
                mx_pred = jnp.mean(mx_loc)
                zz_pred = jnp.mean(zz_loc)

                r_mx = lax.stop_gradient(mx_pred - mx_k)
                r_zz = lax.stop_gradient(zz_pred - zz_k)

                direct = jnp.mean(r_mx * mx_loc)
                x_c    = lax.stop_gradient(mx_loc - mx_pred)
                zz_c   = lax.stop_gradient(zz_loc - zz_pred)
                sf_mx  = jnp.mean(r_mx * x_c  * log_prob)
                sf_zz  = jnp.mean(r_zz * zz_c * log_prob)

                surrogate_k = direct + sf_mx + sf_zz
                mse_k = lax.stop_gradient(r_mx**2 + r_zz**2)
                return carry, (surrogate_k, mse_k)

            _, (surrogates, mses) = lax.scan(
                loss_one_t, None, (M_batch, tgt_mx_b, tgt_zz_b, all_sigmas)
            )
            obs_loss = jnp.sum(surrogates)

            # L2 toward base model weights
            cur_leaves = jax.tree_util.tree_leaves(eqx.filter(model, eqx.is_array))
            reg_loss = reg_lambda * jnp.mean(jnp.stack([
                jnp.mean((c - b) ** 2) for c, b in zip(cur_leaves, base_leaves)
            ]))
            return obs_loss + reg_loss, jnp.mean(mses)

        (_, mse), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(model)
        updates, new_opt_state = optimizer.update(grads, opt_state)
        new_model = eqx.apply_updates(model, updates)
        return new_model, new_opt_state, mse

    rng = random.PRNGKey(1234)
    for _ in tqdm(range(n_steps), desc=f"  FT-decoder [{proto}]", leave=False):
        rng, key_t, key_s = random.split(rng, 3)

        t_batch_idx = random.choice(key_t, T_STEPS, shape=(t_batch,), replace=False)
        tgt_mx_b = tgt_mx_all[t_batch_idx]
        tgt_zz_b = tgt_zz_all[t_batch_idx]

        # On-policy sampling with current model weights
        all_sigmas_list = []
        for i in range(t_batch):
            key_s, sub = random.split(key_s)
            sigmas_k = sample_given_beta(
                sub, model, M_traj[int(t_batch_idx[i])], beta_jnp, n_mc
            )
            all_sigmas_list.append(sigmas_k)
        all_sigmas = jnp.stack(all_sigmas_list, axis=0)

        model, opt_state, mse = ft_step(
            model, opt_state, t_batch_idx, all_sigmas, tgt_mx_b, tgt_zz_b
        )
        mse_val = float(mse)
        mse_hist.append(mse_val)
        if mse_val < best_mse:
            best_mse   = mse_val
            best_model = model

    return best_model, np.array(mse_hist)


In [ ]:
ft_models   = {}
mse_hists_dec = {}

# ── Select which protocols to fine-tune ────────────────────────────────────────
# FT_PROTOCOLS_DEC = PROTOCOLS      # all three
FT_PROTOCOLS_DEC = ["GRF"]          # GRF only
# ───────────────────────────────────────────────────────────────────────────────

for proto in FT_PROTOCOLS_DEC:
    print(f"\n{'='*60}")
    print(f"Fine-tuning decoder: {proto}")
    print(f"{'='*60}")
    t0 = time.time()
    ft_models[proto], mse_hists_dec[proto] = run_protocol_finetune_decoder(
        base_model, proto, dmrg_train
    )
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s  |  "
          f"Final MSE={mse_hists_dec[proto][-1]:.6f}  "
          f"Min MSE={mse_hists_dec[proto].min():.6f}")

# Loss convergence plot
colors = {"GRF": "steelblue", "tanh": "seagreen", "Gaussian": "darkorange"}
fig, axes = plt.subplots(1, len(FT_PROTOCOLS_DEC),
                         figsize=(14 / len(PROTOCOLS) * len(FT_PROTOCOLS_DEC), 3.5),
                         sharey=False)
if len(FT_PROTOCOLS_DEC) == 1:
    axes = [axes]
for ax, proto in zip(axes, FT_PROTOCOLS_DEC):
    ax.semilogy(mse_hists_dec[proto], color=colors[proto], lw=1.5)
    ax.set_title(proto)
    ax.set_xlabel("FT step")
    ax.set_ylabel("MSE (mx + zz, random batch)")
    ax.grid(alpha=0.3)
fig.suptitle("Decoder fine-tuning convergence — M_traj frozen", fontsize=13)
plt.tight_layout()
plt.savefig("ft_decoder_convergence.pdf", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
post_train_dec = {}   # decoder-FT model evaluated on seen state  (seed=42)
post_test_dec  = {}   # decoder-FT model evaluated on unseen state (seed=69)

for proto in FT_PROTOCOLS_DEC:
    print(f"[Post-FT decoder / {proto}]")
    t0 = time.time()
    post_train_dec[proto] = run_mc_benchmark(
        ft_models[proto], dmrg_train[proto],
        n_samples=N_SAMPLES_EVAL, seed=100, verbose=False
    )
    print(f"  seen   done in {time.time()-t0:.1f}s")

    t0 = time.time()
    post_test_dec[proto] = run_mc_benchmark(
        ft_models[proto], dmrg_test[proto],
        n_samples=N_SAMPLES_EVAL, seed=200, verbose=False
    )
    print(f"  unseen done in {time.time()-t0:.1f}s")

# ── Results table ──────────────────────────────────────────────────────────────
OBS_KEYS = ["mx", "mz", "zz"]
header = (f"{'Protocol':<10} {'State':<8}  "
          + "  ".join(f"{'pre '+o:>10} {'post '+o:>10} {chr(0x394)+' '+o:>8}" for o in OBS_KEYS))
print()
print(header)
print("-" * len(header))

for proto in FT_PROTOCOLS_DEC:
    for state_tag, pre_preds, post_preds, refs in [
        ("seen",   pre_train[proto], post_train_dec[proto], dmrg_train[proto]),
        ("unseen", pre_test[proto],  post_test_dec[proto],  dmrg_test[proto]),
    ]:
        row = f"{proto:<10} {state_tag:<8}  "
        for obs in OBS_KEYS:
            pre_mae  = float(np.mean(np.abs(pre_preds[f"{obs}_pred"]  - refs[obs])))
            post_mae = float(np.mean(np.abs(post_preds[f"{obs}_pred"] - refs[obs])))
            row += f"  {pre_mae:>10.5f} {post_mae:>10.5f} {post_mae-pre_mae:>+8.5f}"
        print(row)

# ── GRF plot (if GRF was fine-tuned) ──────────────────────────────────────────
if "GRF" in FT_PROTOCOLS_DEC:
    import matplotlib as mpl
    mpl.rcParams.update({
        "font.family": "serif", "mathtext.fontset": "cm",
        "axes.labelsize": 12, "axes.titlesize": 12,
        "xtick.labelsize": 10, "ytick.labelsize": 10,
        "legend.fontsize": 9,  "lines.linewidth": 1.6,
    })

    proto = "GRF"
    tg    = time_grid * lp.omega_factor

    def dmrg_energy(d):
        return (-lp.J_zz * (N_BONDS / N_SPINS) * d["zz"]
                - d["hz"] * d["mz"] - d["hx"] * d["mx"])

    obs_cfg = [
        ("mx",     r"$\langle m_x\rangle$"),
        ("mz",     r"$\langle m_z\rangle$"),
        ("zz",     r"$\langle zz\rangle$"),
        ("energy", r"$E/N$"),
    ]
    STATE_COLORS = {"seen": "steelblue", "unseen": "tomato"}

    fig, axes = plt.subplots(2, 4, figsize=(17, 7),
                             sharex=True, gridspec_kw={"hspace": 0.12, "wspace": 0.38})

    for row, (state_tag, dmrg_d, pre_res, post_res, state_desc) in enumerate([
        ("seen",   dmrg_train[proto], pre_train[proto], post_train_dec[proto], "seen  (seed=42)"),
        ("unseen", dmrg_test[proto],  pre_test[proto],  post_test_dec[proto],  "UNSEEN (seed=69)"),
    ]):
        color  = STATE_COLORS[state_tag]
        e_dmrg = dmrg_energy(dmrg_d)

        for col, (obs_key, ylabel) in enumerate(obs_cfg):
            ax  = axes[row, col]
            ref = e_dmrg if obs_key == "energy" else dmrg_d[obs_key]
            pre  = pre_res[f"{obs_key}_pred"]
            post = post_res[f"{obs_key}_pred"]

            ax.plot(tg, ref,  color="k",   lw=1.8, label="tDMRG")
            ax.plot(tg, pre,  color=color, lw=1.4, ls="--", alpha=0.55,
                    label=f"pre-FT  (MAE={np.mean(np.abs(pre-ref)):.4f})")
            ax.plot(tg, post, color=color, lw=1.8,
                    label=f"post-FT (MAE={np.mean(np.abs(post-ref)):.4f})")

            if col == 0:
                ax.set_ylabel(ylabel)
            if row == 1:
                ax.set_xlabel(r"$\omega t$")
            if row == 0 and col == 0:
                ax.legend(fontsize=8, loc="best")
            ax.grid(alpha=0.25)

        axes[row, 0].annotate(
            f"Initial state: {state_desc}\n|{dmrg_d['bitstring'][:18]}…⟩",
            xy=(0, 0.5), xycoords="axes fraction",
            xytext=(-0.45, 0.5), textcoords="axes fraction",
            fontsize=9, va="center", rotation=90,
        )

    fig.suptitle(f"Protocol: {proto} — decoder FT — pre- vs post on seen / unseen initial state",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f"ft_decoder_{proto}.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    mpl.rcParams.update(mpl.rcParamsDefault)
